# Categorical Cross-Entropy

This is an optional practice notebook for lecture 1: it is not graded and not part of assignment 1.
Suggested order: mean squared error, then this notebook, then the vanilla training loop.
It computes the categorical cross-entropy for the two models compared on the lecture slides.

The categorical cross-entropy (CCE) is an objective function for *classification* problems with $K$ classes.
It measures how far the class probabilities predicted by a model are from the labels.
For $N$ examples it is defined as

$$
\mathrm{CCE} = -\frac{1}{N} \sum_{n=1}^{N} \sum_{k=1}^{K} y_{nk} \ln(p_{nk}) ,
$$

where $p_{nk}$ is the predicted probability that example $n$ belongs to class $k$, and $y_{nk}$ is the label encoded as a *one-hot* vector.
For example, with the classes (cat, dog, bird), the label "dog" becomes $(0, 1, 0)$.

Collected over all examples, the labels form a matrix $\mathbf{Y} \in \mathbb{R}^{N \times K}$ and the predictions a matrix $\mathbf{P} \in \mathbb{R}^{N \times K}$ whose rows sum to one.
The one-hot label zeroes out all but the correct class, so only the probability assigned to the correct class matters.
The CCE is zero for perfect predictions and grows without bound for confident wrong ones.

In [1]:
import numpy as np

Let's write the CCE as a function.
Every term with $y_{nk} = 0$ vanishes, so the double sum reduces to the log-probability of the correct class of each example.
Selecting that entry directly also avoids computing $\ln(0)$ for classes a model rules out completely.

In [2]:
def categorical_cross_entropy(labels, probs):
    """Return the categorical cross-entropy between one-hot labels and predicted probabilities.

    Args:
        labels: Array of shape `(N, K)` with one-hot rows.
        probs: Array of shape `(N, K)` with predicted class probabilities, rows summing to one.

    Returns:
        The cross-entropy averaged over the `N` examples.
    """
    log_probs_correct = np.log(probs[labels == 1])
    return -np.mean(log_probs_correct)

## The two models from the lecture

Three examples, three classes (cat, dog, bird), labels dog, cat, bird.
Both models predict the same classes, so both have accuracy $2/3$, but Model 2 is more confident where it is right.

In [3]:
labels = np.array([[0, 1, 0], [1, 0, 0], [0, 0, 1]])

probs_model_1 = np.array([[0.3, 0.5, 0.2], [0.6, 0.2, 0.2], [0.1, 0.5, 0.4]])
probs_model_2 = np.array([[0.0, 1.0, 0.0], [0.9, 0.1, 0.0], [0.1, 0.5, 0.4]])

for name, probs in [("Model 1", probs_model_1), ("Model 2", probs_model_2)]:
    accuracy = np.mean(np.argmax(probs, axis=1) == np.argmax(labels, axis=1))
    cce = categorical_cross_entropy(labels, probs)
    print(f"{name}: accuracy = {accuracy:.2f}, cross-entropy = {cce:.2f}")

Model 1: accuracy = 0.67, cross-entropy = 0.71
Model 2: accuracy = 0.67, cross-entropy = 0.34


The accuracy cannot tell the two models apart, but the cross-entropy can: Model 2 wins.
Check the numbers by hand: for Model 1 the correct-class probabilities are $0.5$, $0.6$, $0.4$, so $\mathrm{CCE} = -\frac{1}{3}(\ln(0.5) + \ln(0.6) + \ln(0.4)) \approx 0.71$.

## Near-perfect predictions

If the predicted probabilities are very close to the one-hot labels, the cross-entropy is close to zero.

In [4]:
probs_good = np.array([[0.001, 0.998, 0.001], [0.998, 0.001, 0.001], [0.001, 0.001, 0.998]])
print(f"cross-entropy = {categorical_cross_entropy(labels, probs_good):.4f}")

cross-entropy = 0.0020


## Uniform predictions

A model that assigns the same probability $1/K$ to every class has no idea.
Its cross-entropy is $-\ln(1/K) = \ln(K)$, here $\ln(3) \approx 1.10$.

In [5]:
probs_uniform = np.full((3, 3), 1 / 3)
print(f"cross-entropy = {categorical_cross_entropy(labels, probs_uniform):.4f}")

cross-entropy = 1.0986


## Confidently wrong predictions

A model can do worse than guessing: put almost all probability on a wrong class.

In [6]:
probs_bad = np.array([[0.998, 0.001, 0.001], [0.001, 0.998, 0.001], [0.001, 0.998, 0.001]])
print(f"cross-entropy = {categorical_cross_entropy(labels, probs_bad):.4f}")

cross-entropy = 6.9078


The cross-entropy is much higher because every prediction is confident and wrong.
The logarithm makes a probability near zero on the correct class very expensive.

## Summary

The categorical cross-entropy tells us how good a classifier is.
Unlike the accuracy, it is a *soft* measure: it rewards confident correct predictions gradually, which is what training needs.
It is also known as the negative log-likelihood (NLL).